# 第1章 数据库基础概念

## 📌 学习目标

通过本章学习，你将掌握：
- 什么是数据库和数据库管理系统
- 关系型数据库的核心概念
- 表、行、列的含义
- 主键和外键的作用
- 数据类型基础
- SQL 语言的分类

---

## 1. 什么是数据库？

**数据库（Database）** 是按照一定组织方式存储和管理数据的仓库。

### 生活中的例子

| 场景 | 数据示例 |
|------|----------|
| 银行 | 账户信息、交易记录 |
| 淘宝 | 商品信息、订单、用户 |
| 学校 | 学生信息、课程、成绩 |
| 医院 | 病历、处方、药品 |

### 数据库管理系统（DBMS）

**数据库管理系统** 是用来管理数据库的软件，常见的有：

| DBMS | 特点 |
|------|------|
| **MySQL** | 开源免费，应用最广泛 |
| PostgreSQL | 功能强大，支持复杂数据类型 |
| Oracle | 商业数据库，大型企业使用 |
| SQL Server | 微软产品，与 .NET 集成好 |
| SQLite | 轻量级，嵌入式，无需服务器 |

本教程使用 **MySQL** 进行教学。

## 2. 关系型数据库

**关系型数据库（RDBMS）** 是目前最主流的数据库类型，数据以**表（Table）** 的形式组织。

### 核心概念

```
┌─────────────────────────────────────────────────────┐
│                    数据库 (Database)                  │
│                                                       │
│   ┌──────────────┐  ┌──────────────┐                │
│   │   表 (Table)  │  │   表 (Table)  │   ...          │
│   │              │  │              │                │
│   │  行 (Row)    │  │  行 (Row)    │                │
│   │  行 (Row)    │  │  行 (Row)    │                │
│   │  ...         │  │  ...         │                │
│   └──────────────┘  └──────────────┘                │
│         ↑                                             │
│    列 (Column)                                       │
└─────────────────────────────────────────────────────┘
```

### 2.1 表（Table）

表是数据库中存储数据的基本单位，类似于 Excel 中的工作表。

### 2.2 行（Row）

表中的每一行代表一条**记录（Record）**。

### 2.3 列（Column）

表中的每一列代表一个**字段（Field）**，有特定的数据类型。

## 3. 查看我们的示例表

让我们通过实际操作来理解这些概念：

In [ ]:
import os
import pymysql
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": "shop_db",
    "charset": "utf8mb4",
}

def run_query(sql, params=None):
    conn = pymysql.connect(**DB_CONFIG)
    try:
        return pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

In [ ]:
# 查看 customers 表的全部数据
# 这张表有 5 列（列）和 10 行（记录）
run_query("SELECT * FROM customers")

In [ ]:
# 查看表结构
run_query("DESCRIBE customers")

## 4. 主键（Primary Key）

**主键** 是表中唯一标识每条记录的列（或列组合）。

### 主键的特点

| 特点 | 说明 |
|------|------|
| **唯一性** | 每条记录的主键值都不相同 |
| **非空** | 主键列不允许为 NULL |
| **稳定** | 主键值不应该频繁修改 |

### 例子

在 `customers` 表中，`customer_id` 是主键：
- 张三的 customer_id = 1
- 李四的 customer_id = 2
- 通过 customer_id 可以唯一确定一个客户

### 常见主键设计

| 方式 | 示例 | 优点 | 缺点 |
|------|------|------|------|
| 自增整数 | 1, 2, 3, ... | 简单高效 | 数据迁移时可能冲突 |
| UUID | 'a1b2c3d4-...' | 全球唯一 | 占用空间大，查询慢 |
| 业务编号 | 身份证号、学号 | 有业务含义 | 可能变化 |

In [ ]:
# 主键的唯一性示例
# 每个 customer_id 都是唯一的
run_query("SELECT customer_id, customer_name, city FROM customers")

## 5. 外键（Foreign Key）

**外键** 是一个表中的列，它引用了另一个表的主键，用于建立表与表之间的**关联关系**。

### 例子

```
customers 表                 orders 表
┌──────────────┐           ┌──────────────────┐
│ customer_id  │◄──────────│ customer_id (FK) │
│ customer_name│           │ order_id (PK)    │
│ email        │           │ order_date       │
└──────────────┘           │ total_amount     │
                           └──────────────────┘

PK = Primary Key (主键)
FK = Foreign Key (外键)
```

外键确保了**引用完整性**：订单中的 customer_id 必须在 customers 表中存在。

In [ ]:
# 外键关系示例
# orders 表的 customer_id 引用了 customers 表的 customer_id
run_query("""
    SELECT 
        o.order_id,
        o.customer_id,    -- 外键
        c.customer_name,  -- 通过外键关联到客户名
        o.total_amount
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    LIMIT 5
""")

## 6. 数据类型

每列都有特定的数据类型，决定了该列能存储什么数据。

### MySQL 常用数据类型

#### 数值类型

| 类型 | 说明 | 示例 |
|------|------|------|
| `INT` | 整数 | 1, 100, -5 |
| `BIGINT` | 大整数 | 9999999999 |
| `DECIMAL(M,D)` | 精确小数 | DECIMAL(10,2) → 12345678.90 |
| `FLOAT` | 浮点数（不精确） | 3.14 |

#### 字符串类型

| 类型 | 说明 | 示例 |
|------|------|------|
| `VARCHAR(N)` | 可变长度字符串 | VARCHAR(50) → '张三' |
| `CHAR(N)` | 固定长度字符串 | CHAR(2) → '男' |
| `TEXT` | 长文本 | 文章内容 |

#### 日期时间类型

| 类型 | 说明 | 示例 |
|------|------|------|
| `DATE` | 日期 | 2024-01-15 |
| `DATETIME` | 日期时间 | 2024-01-15 10:30:00 |
| `TIMESTAMP` | 时间戳 | 自动记录修改时间 |

In [ ]:
# 查看各表的数据类型
run_query("DESCRIBE products")

## 7. SQL 语言分类

**SQL（Structured Query Language）** 是操作关系型数据库的标准语言。

### SQL 的四大分类

| 分类 | 全称 | 用途 | 关键字 |
|------|------|------|--------|
| **DDL** | Data Definition Language | 定义数据结构 | CREATE, ALTER, DROP |
| **DML** | Data Manipulation Language | 操作数据 | INSERT, UPDATE, DELETE |
| **DQL** | Data Query Language | 查询数据 | SELECT |
| **DCL** | Data Control Language | 权限控制 | GRANT, REVOKE |

### SQL 语句书写规范

```sql
-- SQL 关键字建议大写（不是必须，但推荐）
SELECT customer_name, city
FROM customers
WHERE city = '北京';

-- 每个子句独占一行
-- 语句以分号 ; 结尾
-- -- 开头是单行注释
```

In [ ]:
# 你的第一条 SQL 查询！
# SELECT 选择哪些列，FROM 从哪张表
run_query("SELECT customer_name, city FROM customers")

## 📝 练习题

### 题目1（简单）
查询 `employees` 表的全部数据。

### 题目2（简单）
查询 `products` 表的结构（查看有哪些列及其数据类型）。

### 题目3（中等）
查询 `orders` 表中所有订单的 `order_id`、`customer_id` 和 `total_amount`。

### 题目4（思考题）
请说出 `customers` 表和 `orders` 表之间是如何通过外键关联的？

---

## ✅ 参考答案

In [ ]:
# 题目1：查询 employees 表的全部数据
run_query("SELECT * FROM employees")

In [ ]:
# 题目2：查看 products 表的结构
run_query("DESCRIBE products")

In [ ]:
# 题目3：查询订单的 order_id, customer_id, total_amount
run_query("SELECT order_id, customer_id, total_amount FROM orders")

**题目4 参考答案：**

`orders` 表中的 `customer_id` 是外键，它引用了 `customers` 表中的 `customer_id`（主键）。

这意味着：
- 每个订单都必须属于一个已存在的客户
- 一个客户可以有多个订单（一对多关系）
- 不能创建一个 customer_id 不存在的订单

---

**下一章：[02_基本查询_SELECT](02_基本查询_SELECT.ipynb)**